# 📮 HTTP POST — Creating Data with FastAPI + Pydantic

**Goal:** understand the HTTP **POST** method, how a client sends data in a **request body**, and how FastAPI + **Pydantic** validate that data
before we save a **new patient record**.

> Continues the previous notebooks: [Pydantic notes](Pydantic_Notes.ipynb) (validation, `Field`, `computed_field`) and [HTTP GET notes](HTTP_get_Concepts.ipynb) (endpoints, status codes, `HTTPException`).

## 📑 Table of Contents
1. What is POST? GET vs POST
2. The big idea: request body → validate → save
3. Setup and imports
4. The Pydantic schema: `Patient`
5. Helper functions: `load_data()` and `save_data()`
6. Sample data (our JSON "database")
7. `POST /create` — the endpoint, line by line
8. Testing every scenario (success, duplicate, invalid data)
9. Issues found in the live-class code (and fixes)
10. Running the real server and using Swagger UI
11. Final consolidated `main.py`
12. Summary, common mistakes, practice exercises

---
# 1️⃣ What is POST?

**POST** is the HTTP method used to **create** a new resource on the server (the **C** in CRUD).
The client sends the new data inside the **request body**, usually as **JSON**, and the server stores it.

| Method | CRUD | Purpose | Body? | Typical success code |
|--------|------|---------|-------|----------------------|
| GET | Read | Fetch data | ❌ No | `200 OK` |
| **POST** | **Create** | **Add a new resource** | ✅ **Yes** | **`201 Created`** |
| PUT | Update | Replace an existing resource | ✅ Yes | `200 OK` |
| PATCH | Update | Change part of a resource | ✅ Yes | `200 OK` |
| DELETE | Delete | Remove a resource | Usually no | `200 OK` / `204 No Content` |

## GET vs POST
| | GET | POST |
|---|-----|------|
| Where does the input go? | In the **URL** (path/query parameters) | In the **request body** (JSON) |
| Changes data on the server? | ❌ Never (safe) | ✅ Yes (creates a record) |
| Idempotent? | ✅ Yes, same result every time | ❌ **No**, sending it twice may create two records (or an error if we forbid duplicates) |
| Data visible in the URL/logs? | Yes | No, the body is not part of the URL |
| Good for sensitive/large data? | ❌ No | ✅ Yes |

### What is a request body?
The part of an HTTP request that carries the data. Example JSON body for a new patient:
```json
{
  "id": "P006",
  "name": "Kabir Singh",
  "city": "Pune",
  "age": 40,
  "gender": "male",
  "height": 1.75,
  "weight": 70
}
```
The client must also send the header **`Content-Type: application/json`** so the server knows how to read it (Swagger, `requests` and `TestClient` set it automatically when you send JSON).

---
# 2️⃣ The big idea (concept taught in class)

> The client sends the data in **JSON format** as the **request body**. We **validate** that data using **Pydantic** and **raise an error** if anything is wrong.
> If everything is valid, we **register it as a new record**.

```
 Client                           FastAPI                                     JSON file / DB
   │  POST /create + JSON body       │                                              │
   │ ───────────────────────────────►│                                              │
   │                                 │ 1. Pydantic validates body against Patient   │
   │  422 Unprocessable Entity  ◄────│    ✗ invalid → automatic error (no code!)    │
   │                                 │ 2. load existing data ─────────────────────► │
   │  400/409 already exists    ◄────│ 3. id already present? → HTTPException       │
   │                                 │ 4. add new record, save ───────────────────► │
   │  201 Created               ◄────│ 5. return success response                   │
```

### Why Pydantic is the star here
When an endpoint function has a parameter whose type is a **Pydantic model**, FastAPI automatically:
1. reads the **JSON body**,
2. **validates** every field (types, ranges, allowed values),
3. **coerces** compatible types (e.g. `"40"` → `40`),
4. returns **422** with a detailed error if validation fails (your function never runs),
5. otherwise gives you a **ready-to-use validated object** (`patient`).

So the code inside the function can *trust* its input. This is exactly the 3-step workflow from the Pydantic notebook, done for us.

---
# 3️⃣ Setup and imports

In [1]:
# %pip install fastapi uvicorn httpx2 "pydantic[email]"     # uncomment if needed (older versions: httpx)

import json
from typing import Annotated, Literal

from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse           # lets us control the status code + body of the response
from fastapi.testclient import TestClient            # a fake client to call our API from the notebook
from pydantic import BaseModel, Field, computed_field, field_validator

app = FastAPI()
client = TestClient(app)

# In this notebook we use a separate demo file so we never overwrite your real patients.json.
# (The final main.py in section 11 uses the real 'patients.json'.)
DATA_FILE = 'patients_demo.json'


def show(response):
    """Helper: print status code and JSON body neatly."""
    print("Status code:", response.status_code)
    print(json.dumps(response.json(), indent=2))

---
# 4️⃣ The Pydantic schema: `Patient`

This model describes **what a valid new patient looks like**. It is the *contract* between the client and our API.

### Concepts used
| Feature | Where | Purpose |
|---------|-------|---------|
| `Annotated[type, Field(...)]` | every field | type + rules + docs in one place |
| `Field(...)` | first argument `...` | the field is **required** |
| `gt=0, lt=120` | `age` | numeric range: `0 < age < 120` |
| `Literal['male','female','other']` | `gender` | only these exact values are allowed |
| `description`, `examples` | all | shown in Swagger docs |
| `@computed_field` + `@property` | `bmi`, `verdict` | values **calculated by the server**; the client does **not** send them |
| `@field_validator(mode='before')` | `gender` | clean the raw input (lower-case) before checking it against the `Literal` |

> 💡 Computed fields are a great security/consistency feature: because the server calculates BMI, a client can never store a wrong BMI.

In [2]:
class Patient(BaseModel):
    # ---------- fields the CLIENT must send ----------
    id: Annotated[str, Field(..., description='ID of the patient', examples=['P001'])]
    name: Annotated[str, Field(..., description='Name of the patient')]
    city: Annotated[str, Field(..., description='City where the patient is living')]
    age: Annotated[int, Field(..., gt=0, lt=120, description='Age of the patient')]
    gender: Annotated[Literal['male', 'female', 'other'], Field(..., description='Gender of the patient')]
    height: Annotated[float, Field(..., gt=0, description='Height of the patient in metres')]
    weight: Annotated[float, Field(..., gt=0, description='Weight of the patient in kg')]

    # ---------- small improvement: accept 'Male', 'FEMALE', ' other ' ----------
    @field_validator('gender', mode='before')
    @classmethod
    def normalize_gender(cls, value):
        # 'before' mode → we receive the RAW input, so clean it first, then the Literal check runs.
        return value.strip().lower() if isinstance(value, str) else value

    # ---------- fields calculated by the SERVER ----------
    @computed_field
    @property
    def bmi(self) -> float:
        # BMI = weight (kg) / height (m)^2
        return round(self.weight / self.height ** 2, 2)

    @computed_field
    @property
    def verdict(self) -> str:
        # Standard BMI categories
        if self.bmi < 18.5:
            return 'Underweight'
        elif self.bmi < 25:
            return 'Normal'
        elif self.bmi < 30:
            return 'Overweight'
        else:
            return 'Obese'

### Try the model on its own (no API yet)

In [3]:
p = Patient(id='P006', name='Kabir Singh', city='Pune', age='40', gender='Male', height=1.75, weight=70)
print(p)                 # note: age '40' → 40, gender 'Male' → 'male', bmi & verdict were computed
print()
print(p.model_dump())    # computed fields ARE included in the dump

id='P006' name='Kabir Singh' city='Pune' age=40 gender='male' height=1.75 weight=70.0 bmi=22.86 verdict='Normal'

{'id': 'P006', 'name': 'Kabir Singh', 'city': 'Pune', 'age': 40, 'gender': 'male', 'height': 1.75, 'weight': 70.0, 'bmi': 22.86, 'verdict': 'Normal'}


---
# 5️⃣ Helper functions: `load_data()` and `save_data()`

Our "database" is a JSON file. We need two helpers:

| Function | Mode | What it does |
|----------|------|--------------|
| `load_data()` | `'r'` (read) | JSON file → Python `dict` (`json.load`) |
| `save_data(data)` | `'w'` (write) | Python `dict` → JSON file (`json.dump`). ⚠️ `'w'` **overwrites the whole file**, so we must always write the *complete* updated dictionary |

`with open(...)` closes the file automatically, even if an error happens.

> 📝 The live-class function was named `save_date` (a typo). It is renamed to **`save_data`**.

In [4]:
def load_data():
    """Read all patients from the JSON file and return them as a dict."""
    with open(DATA_FILE, 'r') as f:
        return json.load(f)


def save_data(data):
    """Write the complete patients dict back to the JSON file (overwrites the file)."""
    with open(DATA_FILE, 'w') as f:
        json.dump(data, f, indent=4)      # indent=4 → human-readable file

---
# 6️⃣ Sample data (our JSON "database")

We create the starting data **using the `Patient` model itself**, so the file has exactly the shape our API will produce later:
- the **id is the key** of the dictionary,
- the record holds the other fields **plus** `bmi` and `verdict`.

> ⚠️ Running this cell **resets** `patients_demo.json` to 3 patients, so every run of the notebook starts clean.

In [5]:
starting_patients = [
    Patient(id='P001', name='Ananya Verma', city='Delhi',  age=28, gender='female', height=1.65, weight=90),
    Patient(id='P002', name='Rohan Mehta',  city='Mumbai', age=35, gender='male',   height=1.80, weight=72),
    Patient(id='P003', name='Sita Sharma',  city='Kathmandu', age=42, gender='female', height=1.58, weight=48),
]

# {id: record_without_id}
initial_data = {p.id: p.model_dump(exclude={'id'}) for p in starting_patients}
save_data(initial_data)

print("Patients in DB:", list(load_data().keys()))
print(json.dumps(load_data()['P001'], indent=2))

Patients in DB: ['P001', 'P002', 'P003']
{
  "name": "Ananya Verma",
  "city": "Delhi",
  "age": 28,
  "gender": "female",
  "height": 1.65,
  "weight": 90.0,
  "bmi": 33.06,
  "verdict": "Obese"
}


---
# 7️⃣ `POST /create` — the endpoint, line by line

```python
@app.post('/create')
def create_patient(patient: Patient):
```
- `@app.post('/create')` → this function handles **POST** requests to `/create`.
- `patient: Patient` → the parameter type is a **Pydantic model**, so FastAPI reads the **JSON request body** and validates it against `Patient`.

### The 4 steps inside the function
| Step | Code | Why |
|------|------|-----|
| 1. Load | `data = load_data()` | get the current records |
| 2. Check duplicate | `if patient.id in data: raise HTTPException(...)` | ids must be unique |
| 3. Add | `data[patient.id] = patient.model_dump(exclude={'id'})` | the id becomes the **key**, so we exclude it from the value |
| 4. Save | `save_data(data)` | persist to the file |
| 5. Respond | `JSONResponse(status_code=201, content=...)` | tell the client it was created |

### Why `JSONResponse`?
By default FastAPI answers with status **200**. For a successful creation the correct code is **201 Created**, so we build the response ourselves with `JSONResponse(status_code=201, content=...)`.
(Alternative: `@app.post('/create', status_code=201)` and simply `return {...}`.)

### Which error for a duplicate id?
| Code | Meaning |
|------|---------|
| `400 Bad Request` | generic "your request is wrong" (used in class) |
| `409 Conflict` | **more precise**: the request conflicts with the current state (the id already exists) |

We use **409**. Either is accepted in practice, but 409 tells the client exactly what happened.

In [6]:
@app.post('/create')
def create_patient(patient: Patient):
    # At this point Pydantic has ALREADY validated the body. Invalid data never reaches here (→ 422).

    # 1) Load the existing data
    data = load_data()

    # 2) Business rule: the id must be unique
    if patient.id in data:
        raise HTTPException(status_code=409, detail='Patient already exists.')

    # 3) Add the new patient. Key = id, value = all other fields (incl. computed bmi & verdict)
    data[patient.id] = patient.model_dump(exclude={'id'})

    # 4) Save the complete updated dictionary back to the file
    save_data(data)

    # 5) Respond with 201 Created
    return JSONResponse(
        status_code=201,
        content={'message': 'Patient created successfully', 'id': patient.id},
    )

---
# 8️⃣ Testing every scenario

With `TestClient`, `client.post(url, json=body)` sends a POST request and automatically sets `Content-Type: application/json`.

### ✅ Scenario 1: valid new patient → `201 Created`

In [7]:
new_patient = {
    'id': 'P006',
    'name': 'Kabir Singh',
    'city': 'Pune',
    'age': 40,
    'gender': 'male',
    'height': 1.75,
    'weight': 70,
}

show(client.post('/create', json=new_patient))

Status code: 201
{
  "message": "Patient created successfully",
  "id": "P006"
}


Let's confirm it was really **saved** in the file, including the computed `bmi` and `verdict`:

In [8]:
data = load_data()
print("Patients now:", list(data.keys()))
print(json.dumps(data['P006'], indent=2))     # note: no 'id' inside the record, it is the key

Patients now: ['P001', 'P002', 'P003', 'P006']
{
  "name": "Kabir Singh",
  "city": "Pune",
  "age": 40,
  "gender": "male",
  "height": 1.75,
  "weight": 70.0,
  "bmi": 22.86,
  "verdict": "Normal"
}


### ❌ Scenario 2: same id again → `409 Conflict`
POST is **not idempotent**: sending the same request twice does *not* give the same result. The second time our duplicate check rejects it.

In [9]:
show(client.post('/create', json=new_patient))

Status code: 409
{
  "detail": "Patient already exists."
}


### ❌ Scenario 3: invalid data → `422 Unprocessable Entity` (automatic!)
We wrote **no** `if` statements for these; Pydantic does all of it. Look at the `loc` field: it tells us exactly *where* the error is (`body → age`).

**3a. Age out of range** (`gt=0, lt=120`)

In [10]:
bad = {**new_patient, 'id': 'P007', 'age': 150}
show(client.post('/create', json=bad))

Status code: 422
{
  "detail": [
    {
      "type": "less_than",
      "loc": [
        "body",
        "age"
      ],
      "msg": "Input should be less than 120",
      "input": 150,
      "ctx": {
        "lt": 120
      }
    }
  ]
}


**3b. Gender not in the allowed list** (`Literal`)

In [11]:
bad = {**new_patient, 'id': 'P007', 'gender': 'robot'}
show(client.post('/create', json=bad))

Status code: 422
{
  "detail": [
    {
      "type": "literal_error",
      "loc": [
        "body",
        "gender"
      ],
      "msg": "Input should be 'male', 'female' or 'other'",
      "input": "robot",
      "ctx": {
        "expected": "'male', 'female' or 'other'"
      }
    }
  ]
}


**3c. Missing required field** (`name` not sent)

In [12]:
bad = {k: v for k, v in new_patient.items() if k != 'name'}
bad['id'] = 'P007'
show(client.post('/create', json=bad))

Status code: 422
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "name"
      ],
      "msg": "Field required",
      "input": {
        "id": "P007",
        "city": "Pune",
        "age": 40,
        "gender": "male",
        "height": 1.75,
        "weight": 70
      }
    }
  ]
}


**3d. Wrong type** (`weight` is text that cannot become a number)

In [13]:
bad = {**new_patient, 'id': 'P007', 'weight': 'heavy'}
show(client.post('/create', json=bad))

Status code: 422
{
  "detail": [
    {
      "type": "float_parsing",
      "loc": [
        "body",
        "weight"
      ],
      "msg": "Input should be a valid number, unable to parse string as a number",
      "input": "heavy"
    }
  ]
}


### ✅ Scenario 4: type coercion + cleaning
`age` sent as text `"29"` is coerced to `29`, and `"FEMALE "` is cleaned by our `before` validator. The server also **ignores** a fake `bmi` sent by the client and calculates its own.

In [14]:
messy = {
    'id': 'P007', 'name': 'Meera Rao', 'city': 'Jaipur',
    'age': '29',              # str → int
    'gender': ' FEMALE ',     # cleaned → 'female'
    'height': 1.60, 'weight': 55,
    'bmi': 999,               # client tries to fake BMI → extra field is ignored
}

show(client.post('/create', json=messy))
print(json.dumps(load_data()['P007'], indent=2))    # bmi is 21.48 (server-calculated), not 999

Status code: 201
{
  "message": "Patient created successfully",
  "id": "P007"
}
{
  "name": "Meera Rao",
  "city": "Jaipur",
  "age": 29,
  "gender": "female",
  "height": 1.6,
  "weight": 55.0,
  "bmi": 21.48,
  "verdict": "Normal"
}


### ❌ Scenario 5: wrong method → `405`
`/create` only accepts POST. A GET request is rejected.

In [15]:
show(client.get('/create'))

Status code: 405
{
  "detail": "Method Not Allowed"
}


---
# 9️⃣ Issues found in the live-class code (and fixes)

| # | Live-class code | Problem | Fix |
|---|-----------------|---------|-----|
| 1 | `elif self.bmi < 30: return 'Normal'` | **Wrong BMI category.** A BMI of 27 is *Overweight*, not *Normal*. `Normal` should end at 25. | `< 18.5` Underweight, `< 25` Normal, `< 30` Overweight, else Obese |
| 2 | `def save_date(data)` | Typo in the function name (`date` instead of `data`). | `save_data` |
| 3 | `JSONResponse(status_code=201, content='patient created successfully')` | Content is a bare string. It works, but APIs should return a **JSON object** clients can extend/parse. | `content={'message': ..., 'id': ...}` |
| 4 | `status_code=400` for a duplicate | Works, but `409 Conflict` describes the situation precisely. | `409` |
| 5 | `model_dump(exclude=['id'])` | `exclude` is documented to take a **set** (or dict). A list happens to work but is not the intended type. | `exclude={'id'}` |
| 6 | `gender` accepts only exact lower-case | `'Male'` (capital M) is rejected, which is annoying for real users. | `field_validator('gender', mode='before')` to normalize |
| 7 | `json.dump(data, f)` | File is written on one line, hard to read/debug. | `indent=4` |

Let's *prove* issue #1 with a BMI of about 27:

In [16]:
bmi = 27
old_verdict = 'Underweight' if bmi < 18.5 else 'Normal' if bmi < 30 else 'Obese'                        # live-class logic
new_verdict = 'Underweight' if bmi < 18.5 else 'Normal' if bmi < 25 else 'Overweight' if bmi < 30 else 'Obese'  # fixed

print("BMI 27 → old:", old_verdict, "| fixed:", new_verdict)

BMI 27 → old: Normal | fixed: Overweight


### ⚠️ Limitations of a JSON file as a database (good to know)
- Every request reads and rewrites the **whole file**, which is slow for big data.
- Two requests writing at the same moment can **overwrite each other** (a *race condition*).
- For real projects use a proper database (SQLite / PostgreSQL / MongoDB) with a library like SQLAlchemy.
  The **POST logic stays identical**: validate → check → insert → return 201.

---
# 🔟 Running the real server and using Swagger UI

Save the final code as `main.py` (section 11) and run in a **terminal**:
```bash
uvicorn main:app --reload
```
Open **`http://127.0.0.1:8000/docs`** (Swagger UI):
1. Click **`POST /create`** → **Try it out**.
2. Swagger shows an editable JSON body generated from the `Patient` schema (fields marked as *required*, with descriptions and the `P001` example).
3. Edit the JSON → **Execute** → see the status code (`201`, `409` or `422`) and the response.

### Calling it from Python (`requests`)
```python
import requests

body = {"id": "P010", "name": "Test User", "city": "Delhi",
        "age": 30, "gender": "male", "height": 1.7, "weight": 65}

r = requests.post("http://127.0.0.1:8000/create", json=body)   # json= sets Content-Type automatically
print(r.status_code, r.json())
```

### Calling it with `curl`
```bash
curl -X POST http://127.0.0.1:8000/create \
     -H "Content-Type: application/json" \
     -d '{"id":"P010","name":"Test User","city":"Delhi","age":30,"gender":"male","height":1.7,"weight":65}'
```
> 💡 A browser address bar can only send GET requests, so to test POST use Swagger UI, `requests`, `curl` or Postman.

---
# 1️⃣1️⃣ Final consolidated `main.py`

The cell writes the complete, corrected API (with the GET endpoints from the previous notebook + the new POST endpoint) to `main.py`.
It uses the real file name **`patients.json`**.

In [17]:
%%writefile main.py
import json
from typing import Annotated, Literal

from fastapi import FastAPI, Path, Query, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, computed_field, field_validator

app = FastAPI()


# ───────────────────────── Pydantic schema ─────────────────────────
class Patient(BaseModel):
    id: Annotated[str, Field(..., description='ID of the patient', examples=['P001'])]
    name: Annotated[str, Field(..., description='Name of the patient')]
    city: Annotated[str, Field(..., description='City where the patient is living')]
    age: Annotated[int, Field(..., gt=0, lt=120, description='Age of the patient')]
    gender: Annotated[Literal['male', 'female', 'other'], Field(..., description='Gender of the patient')]
    height: Annotated[float, Field(..., gt=0, description='Height of the patient in metres')]
    weight: Annotated[float, Field(..., gt=0, description='Weight of the patient in kg')]

    @field_validator('gender', mode='before')
    @classmethod
    def normalize_gender(cls, value):
        # Accept 'Male', 'FEMALE', ' other ' etc.
        return value.strip().lower() if isinstance(value, str) else value

    @computed_field
    @property
    def bmi(self) -> float:
        return round(self.weight / self.height ** 2, 2)

    @computed_field
    @property
    def verdict(self) -> str:
        if self.bmi < 18.5:
            return 'Underweight'
        elif self.bmi < 25:
            return 'Normal'
        elif self.bmi < 30:
            return 'Overweight'
        return 'Obese'


# ───────────────────────── Data helpers ─────────────────────────
def load_data():
    """Read all patients from the JSON file."""
    with open('patients.json', 'r') as f:
        return json.load(f)


def save_data(data):
    """Write the complete patients dict back to the JSON file."""
    with open('patients.json', 'w') as f:
        json.dump(data, f, indent=4)


# ───────────────────────── GET endpoints ─────────────────────────
@app.get('/')
def hello():
    return {'message': 'Patient Management System API'}


@app.get('/view')
def view():
    return load_data()


@app.get('/patient/{patient_id}')
def view_patient(patient_id: str = Path(..., description='ID of the patient in the DB', examples=['P001'])):
    data = load_data()
    if patient_id in data:
        return data[patient_id]
    raise HTTPException(status_code=404, detail='Patient not found')


@app.get('/sort')
def sort_patients(
    sort_by: str = Query(..., description='Sort on the basis of height, weight or bmi'),
    order: str = Query('asc', description='Sort in asc or desc order'),
):
    valid_fields = ['height', 'weight', 'bmi']
    if sort_by not in valid_fields:
        raise HTTPException(status_code=400, detail=f'Invalid field, select from {valid_fields}')
    if order not in ['asc', 'desc']:
        raise HTTPException(status_code=400, detail='Invalid order, select either asc or desc')

    data = load_data()
    return sorted(data.values(), key=lambda x: x.get(sort_by, 0), reverse=(order == 'desc'))


# ───────────────────────── POST endpoint ─────────────────────────
@app.post('/create')
def create_patient(patient: Patient):
    """Validate the JSON body with Pydantic, then add the new patient."""
    data = load_data()

    # ids must be unique
    if patient.id in data:
        raise HTTPException(status_code=409, detail='Patient already exists.')

    # id is the key, so leave it out of the stored record
    data[patient.id] = patient.model_dump(exclude={'id'})
    save_data(data)

    return JSONResponse(status_code=201, content={'message': 'Patient created successfully', 'id': patient.id})

Overwriting main.py


---
# 1️⃣2️⃣ Summary

## 🧠 Cheat-sheet
| Concept | Code | Notes |
|---------|------|-------|
| POST endpoint | `@app.post('/create')` | used to **create** data |
| Request body | parameter typed as a Pydantic model: `patient: Patient` | FastAPI reads + validates the JSON body |
| Required field | `Field(...)` | the client must send it |
| Range / choices | `gt`, `lt`, `Literal[...]` | invalid → automatic **422** |
| Server-calculated values | `@computed_field` + `@property` | client cannot fake them |
| Clean raw input | `@field_validator(..., mode='before')` | normalize before validation |
| Duplicate check | `raise HTTPException(status_code=409, ...)` | business rule |
| Object → dict | `patient.model_dump(exclude={'id'})` | ready to store as JSON |
| Success response | `JSONResponse(status_code=201, content={...})` | **201 Created** |
| Test in notebook | `client.post('/create', json=body)` | no server needed |

## 🔑 Key ideas
1. **POST sends data in the body**, GET sends it in the URL.
2. **Pydantic validates automatically**: bad data → **422**, and your function only ever sees clean, valid data.
3. **Two layers of validation:** Pydantic for the *shape* of the data (types, ranges), and your own `if`/`HTTPException` for *business rules* (unique id).
4. **Success = `201 Created`**, not the default 200.
5. POST is **not idempotent**, so plan for duplicates.

## ⚠️ Common mistakes
1. Forgetting to `save_data(data)` (the record disappears when the request ends).
2. Saving only the new patient instead of the **whole** dictionary. `'w'` mode overwrites the file, so all old patients would be lost.
3. Trying to test POST by typing the URL in the browser (that sends GET → 405).
4. Sending form fields or query params instead of a **JSON body**.
5. Storing the `id` twice (as key and inside the record). Use `exclude={'id'}`.
6. Trusting the client for calculated values (BMI). Compute them on the server.
7. Wrong BMI thresholds or a typo in the helper name (`save_date`).

## 🎯 Practice exercises
1. Add `POST /create` validation so that `name` must be at least 2 characters (`min_length=2`).
2. Return the **full created record** (with `bmi` and `verdict`) in the 201 response.
3. Make the new id automatic: generate `P007`, `P008`, ... instead of asking the client to send `id`.
4. Add `PUT /edit/{patient_id}` to update a patient (**U** in CRUD) and `DELETE /delete/{patient_id}` (**D** in CRUD).
5. Add a `model_validator` rejecting patients with `age < 18` and `verdict == 'Obese'` (invent your own rule) and see the 422 message.
6. Replace the JSON file with an SQLite database using `sqlite3` while keeping the same endpoint code shape.

## 🔗 References
- FastAPI request body: <https://fastapi.tiangolo.com/tutorial/body/>
- FastAPI response status codes: <https://fastapi.tiangolo.com/tutorial/response-status-code/>
- Pydantic docs: <https://docs.pydantic.dev/latest/>
- MDN: POST method: <https://developer.mozilla.org/en-US/docs/Web/HTTP/Methods/POST>